In [44]:
orders = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("abfss://bronze@stzomatoanalytics01.dfs.core.windows.net/raw/orders.csv")

In [45]:
print("Rows:", orders.count())

orders.selectExpr(
    "count(distinct user_id) as users",
    "count(distinct r_id) as restaurants"
).show()

In [47]:
from pyspark.sql.functions import *

fact_orders = (
    orders
    .withColumnRenamed("_c0", "order_id")
    .withColumn("restaurant_id", col("r_id").cast("long"))
    .drop("r_id")
)

In [48]:
fact_orders = fact_orders.filter(
    col("restaurant_id").isNotNull()
)
fact_orders.count()

In [49]:
fact_orders.groupBy("currency").count().show()

In [50]:
from pyspark.sql.functions import sum, when

fact_orders.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in fact_orders.columns
]).show()

In [51]:
fact_orders.select("order_id").distinct().count()

In [52]:
fact_orders.printSchema()

In [53]:
fact_orders = fact_orders.withColumnRenamed(
    "Unnamed: 0",
    "order_id"
)

In [54]:
fact_orders.printSchema()

In [56]:
fact_orders.select("order_id").distinct().count()


In [57]:
fact_orders.count()

In [59]:
fact_orders.write \
    .mode("overwrite") \
    .parquet(
        "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/fact_orders"
    )

In [60]:
fact_orders_silver = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/fact_orders"
)

print("Rows:", fact_orders_silver.count())

fact_orders_silver.printSchema()

fact_orders_silver.show(5)